# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuguda999/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one (client, content item, day)** — a single page's single day of measured search performance. Grain: `report_date + client_hash_id + content_hash_id` in `fact_content_daily_performance`.

**Table(s):** `fact_content_daily_performance`, `month=2026-03` partition — a mid-panel month, picked on purpose. The sealed final month (June 2026) is never touched during development, per the iteration rule. `dim_content` is the join target for static metadata; referenced but not pulled in this pass.

**Time window:** `report_date` 2026-03-01 → 2026-03-31, split internally at the midpoint: days 1–15 become the **feature window**, days 16–31 become the **label window** — a genuine past→future split inside one mid-panel month, not the starter CSV's same-window proxy.

Verified below.

In [1]:
import os
import getpass
import duckdb

# Token order: env var -> Colab Secret -> prompt (last resort). Never hardcode — repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
con.execute("SET http_timeout=300")  # seconds; default is 30s, too short for the heavier queries below
con.execute("SET http_retries=5")
con.execute("SET http_retry_wait_ms=1000")
con.execute("SET http_retry_backoff=2")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

# schema only (metadata read, no data pull) — confirms the grain columns exist
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{MONTH}')").df()[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 2. Fields: feature / label / context / excluded

- **Context** (grouping/joining only, never a feature): `client_hash_id`, `content_hash_id` — pseudonyms; `report_date` — used only to build the two windows.
- **Feature** (all computed ONLY from days 1–15, the feature window):
  - `imp_prev` — total `gsc_impressions`, days 1–15
  - `clk_prev` — total `gsc_clicks`, days 1–15
  - `ctr_prev` — `clk_prev / imp_prev`
  - `avg_position_prev` — mean `gsc_avg_position` where position > 0, days 1–15
  - `active_days_prev` — count of days with impressions > 0, days 1–15
- **Label / proxy:** `is_declining = imp_last < 0.8 * imp_prev`, where `imp_last` is total `gsc_impressions` in days 16–31 (the label window). `imp_last` itself is never a feature — see the trap below for exactly why.
- **Excluded:** GA4 columns (`ga4_sessions`, `ga4_engaged_sessions`, `scroll_events`, …) for this pass — only a small share of March rows have `ga4_data_available = TRUE` (verified in section 3), so a raw GA4 feature would mostly encode "does this client have GA4 wired up," not real engagement. Revisit once I build a proper `has_ga4`-gated feature.

In [2]:
import pandas as pd

field_map = pd.DataFrame([
    ("client_hash_id", "context", "join/group key only"),
    ("content_hash_id", "context", "join/group key only"),
    ("report_date", "context", "used to build the two windows, not fed to the model"),
    ("imp_prev", "feature", "SUM(gsc_impressions), days 1-15"),
    ("clk_prev", "feature", "SUM(gsc_clicks), days 1-15"),
    ("ctr_prev", "feature", "clk_prev / imp_prev"),
    ("avg_position_prev", "feature", "AVG(gsc_avg_position) where position > 0, days 1-15"),
    ("active_days_prev", "feature", "COUNT DISTINCT days with impressions > 0, days 1-15"),
    ("imp_last", "label input", "SUM(gsc_impressions), days 16-31 -- builds is_declining, never a feature"),
    ("is_declining", "label", "imp_last < 0.8 * imp_prev"),
    ("ga4_* columns", "excluded", "too sparse this month to trust as a feature (see section 3)"),
], columns=["field", "bucket", "definition / why"])
field_map

,field,bucket,definition / why
0,client_hash_id,context,join/group key only
1,content_hash_id,context,join/group key only
2,report_date,context,"used to build the two windows, not fed to the ..."
3,imp_prev,feature,"SUM(gsc_impressions), days 1-15"
4,clk_prev,feature,"SUM(gsc_clicks), days 1-15"
5,ctr_prev,feature,clk_prev / imp_prev
6,avg_position_prev,feature,"AVG(gsc_avg_position) where position > 0, days..."
7,active_days_prev,feature,"COUNT DISTINCT days with impressions > 0, days..."
8,imp_last,label input,"SUM(gsc_impressions), days 16-31 -- builds is_..."
9,is_declining,label,imp_last < 0.8 * imp_prev


## 3. Verify it with queries (grain, counts, missing values, windows)

Three small queries, one mid-panel month (`month=2026-03` — never the sealed final month). Then the five-feature frame, then the leakage trap.

### Query 1 — grain

In [3]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM read_parquet('{MONTH}')
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print(f"duplicate keys found: {len(grain_check)}  (0 means the grain holds)")
grain_check

duplicate keys found: 0  (0 means the grain holds)


,report_date,client_hash_id,content_hash_id,c


### Query 2 — row count + date span

In [4]:
counts = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{MONTH}')
""").df()
counts

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — availability (`IS TRUE`)

In [5]:
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
           ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_ga4_available
    FROM read_parquet('{MONTH}')
""").df()
availability

,total_rows,ga4_available_rows,pct_ga4_available
0,9841378,413966,4.2


### Five features (max) — built from the days 1–15 feature window

Each is knowable at the decision moment (day 16 of the month) because it's aggregated ONLY over days already elapsed:

1. `imp_prev` — available because it's a closed sum over days already elapsed.
2. `clk_prev` — same: a closed sum over days already elapsed.
3. `ctr_prev` — a ratio of two already-elapsed sums.
4. `avg_position_prev` — an average of already-observed daily positions.
5. `active_days_prev` — a count of already-elapsed days, known the moment they pass.

`imp_last` (days 16–31) is NOT a feature — it only exists after the fact, which is exactly why the trap below uses it.

In [6]:
feat = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_prev,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_prev,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_prev,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN report_date END) AS active_days_prev,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_last
    FROM read_parquet('{MONTH}')
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) > 0
""").df()

feat["ctr_prev"] = feat["clk_prev"] / feat["imp_prev"]
feat["is_declining"] = (feat["imp_last"] < 0.8 * feat["imp_prev"]).astype(int)
feat = feat.dropna(subset=["avg_position_prev"])

print(f"shape: {feat.shape[0]:,} rows x {feat.shape[1]} cols  |  one row = one (client, content item) with real March impressions")
print(f"is_declining positive rate: {feat['is_declining'].mean() * 100:.1f}%")
feat.head(5)

shape: 150,675 rows x 9 cols  |  one row = one (client, content item) with real March impressions
is_declining positive rate: 32.6%


,client_hash_id,content_hash_id,imp_prev,clk_prev,avg_position_prev,active_days_prev,imp_last,ctr_prev,is_declining
0,client_3ffa76342f366962,content_749ecd5005b82915,1.0,0.0,1.00000,1,5.0,0.000000,0
1,client_3ffa76342f366962,content_3bc727ad6e986d23,4.0,0.0,15.00000,4,0.0,0.000000,1
2,client_3ffa76342f366962,content_fe3be52ee6f12088,2.0,0.0,3.50000,1,0.0,0.000000,1
3,client_3ffa76342f366962,content_250a26997806cdb7,24.0,1.0,5.90625,8,1.0,0.041667,1
4,client_3ffa76342f366962,content_4c65659252a4d124,42.0,1.0,6.19537,10,21.0,0.023810,1


### The trap: one label-derived column

`is_declining` is defined by `imp_last < 0.8 * imp_prev`. Adding the exact ratio behind that rule as a "feature" should make the model unrealistically good — because it IS the rule, just rearranged.

In [7]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["imp_prev", "clk_prev", "ctr_prev", "avg_position_prev", "active_days_prev"]
X, y = feat[honest_features], feat["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"Honest ROC-AUC (5 features only): {honest_auc:.3f}")

# --- THE TRAP: add the exact ratio the label is computed from ---
feat["future_ratio_LEAKY"] = feat["imp_last"] / feat["imp_prev"]
X_leaky = feat[honest_features + ["future_ratio_LEAKY"]]
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)

leaky_model = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X_tr_l, y_tr_l)
leaky_auc = roc_auc_score(y_te_l, leaky_model.predict_proba(X_te_l)[:, 1])
print(f"'Leaky' ROC-AUC (+ future_ratio_LEAKY): {leaky_auc:.3f}  <- looks amazing")
print(export_text(leaky_model, feature_names=honest_features + ["future_ratio_LEAKY"]))

Honest ROC-AUC (5 features only): 0.634
'Leaky' ROC-AUC (+ future_ratio_LEAKY): 1.000  <- looks amazing
|--- future_ratio_LEAKY <= 0.80
|   |--- class: 1
|--- future_ratio_LEAKY >  0.80
|   |--- class: 0



The tree just splits on `future_ratio_LEAKY <= 0.80` and nails the label — because that ratio IS the label, rearranged. That's leakage: the feature is the answer in disguise. Deleting it and keeping the honest number:

In [8]:
feat = feat.drop(columns=["future_ratio_LEAKY"])
print(f"Leaky column removed. Columns kept: {list(feat.columns)}")
print(f"Honest ROC-AUC to report going forward: {honest_auc:.3f}")

Leaky column removed. Columns kept: ['client_hash_id', 'content_hash_id', 'imp_prev', 'clk_prev', 'avg_position_prev', 'active_days_prev', 'imp_last', 'ctr_prev', 'is_declining']
Honest ROC-AUC to report going forward: 0.634


## 4. Data limits

**Named limitation: GA4 coverage is too thin this month to use.** Only a small share of March rows have `ga4_data_available = TRUE` (Query 3 above) — most clients in this snapshot aren't GA4-wired for this period, so any GA4-based feature would mostly encode "which client is this" rather than real engagement signal. Excluded for now.

Other limits of this slice:
- **One month, one client mix.** March 2026 alone can't separate a real per-page decline from a site-wide seasonal dip or a SERP-wide event that month — that needs a look across multiple months (the lane guide's consolidation/seasonality/noise checks).
- **Unbalanced panel.** Client history depth varies (`dim_clients.gsc_data_start` differs per client); I didn't check per-client depth for March specifically here, so some clients may be thinly represented in this partition.
- **Same-month split, not true multi-month forecasting.** Days 1–15 → days 16–31 is a genuine past→future split, but it's still one 31-day window. A stronger version predicts across month boundaries (e.g. February features → March label).

In [9]:
print(f"clients represented in this slice: {feat['client_hash_id'].nunique()}")
print("Per-client history depth not re-verified for March specifically here — see "
      "docs/ml-intern-dataset-and-lane-guide.md section 2 for the documented unbalanced-panel "
      "numbers (9 of 70 clients have 12+ months of history; a third have little or no usable history).")

clients represented in this slice: 44
Per-client history depth not re-verified for March specifically here — see docs/ml-intern-dataset-and-lane-guide.md section 2 for the documented unbalanced-panel numbers (9 of 70 clients have 12+ months of history; a third have little or no usable history).


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.